In [1]:
from crosscode.trainers.topk_crosscoder.run import *

/home/tim/crosscoder_training/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import yaml
with open('/home/tim/crosscoder_training/crosscode/trainers/topk_crosscoder/gemma_configs.yaml') as f:
    cfg = yaml.safe_load(f)
cfg = TopKAcausalCrosscoderExperimentConfig(**cfg)

In [7]:
device = get_device()

llms = build_llms(
    cfg.data.activations_harvester.llms,
    cfg.cache_dir,
    device,
    inferenced_type=cfg.data.activations_harvester.inference_dtype,
)

match cfg.train.topk_style:
    case "topk":
        cc_act = TopkActivation(k=cfg.crosscoder.k)
    case "batch_topk":
        cc_act = BatchTopkActivation(k_per_example=cfg.crosscoder.k)
    case "groupmax":
        cc_act = GroupMaxActivation(k_groups=cfg.crosscoder.k, latents_size=cfg.crosscoder.n_latents)

d_model = llms[0].cfg.d_model

crosscoder = ModelHookpointAcausalCrosscoder(
    n_models=len(llms),
    n_hookpoints=len(cfg.hookpoints),
    d_model=d_model,
    n_latents=cfg.crosscoder.n_latents,
    init_strategy=AnthropicTransposeInit(dec_init_norm=cfg.crosscoder.dec_init_norm),
    activation_fn=cc_act,
    use_encoder_bias=cfg.crosscoder.use_encoder_bias,
    use_decoder_bias=cfg.crosscoder.use_decoder_bias,
)

Loading checkpoint shards: 100%|██████████| 3/3 [00:00<00:00,  5.43it/s]


Loaded pretrained model google/gemma-2-2b into HookedTransformer


2025-04-09 22:31:49 - INFO - Assigned model key: tl-google_gemma-2-2b to model gemma-2-2b


Moving model to device:  cuda


In [8]:
crosscoder.W_dec_LMPD.shape

torch.Size([20000, 1, 3, 2304])

In [9]:
from einops import rearrange
import torch
encfull = rearrange(crosscoder.W_dec_LMPD.clone(), "l ... -> ... l")
torch.abs(crosscoder.W_enc_MPDL- encfull).max()

tensor(0., grad_fn=<MaxBackward1>)

In [11]:
enc_1 = crosscoder.W_enc_MPDL[0,0]
dec_1 = crosscoder.W_dec_LMPD[:,0,0,:]

In [14]:
torch.allclose(dec_1.T, enc_1)

True

In [1]:
from crosscode.models.acausal_crosscoder import *
inited_cc = ModelHookpointAcausalCrosscoder.load('/home/tim/crosscoder_training/.checkpoints/less_layers_w_n_dead_latents_2025-04-09_21-29-32/epoch_0_step_0', 'cuda')

/home/tim/crosscoder_training/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
inited_cc.W_dec_LMPD.shape

torch.Size([20000, 1, 3, 2304])

In [4]:
inited_cc.W_enc_MPDL.shape

torch.Size([1, 3, 2304, 20000])

In [10]:
from einops import rearrange
encfull = rearrange(inited_cc.W_dec_LMPD.clone(), "l ... -> ... l")

In [12]:
torch.abs(inited_cc.W_enc_MPDL- encfull).max()

tensor(0.0516, grad_fn=<MaxBackward1>)

In [5]:
enc_1 = inited_cc.W_enc_MPDL[0,0]
dec_1 = inited_cc.W_dec_LMPD[:,0,0,:]

In [8]:
dec_1.T

tensor([[ 0.0041,  0.0060, -0.0025,  ..., -0.0100, -0.0117, -0.0026],
        [-0.0110, -0.0097, -0.0175,  ...,  0.0005, -0.0090, -0.0067],
        [-0.0100, -0.0103,  0.0040,  ..., -0.0084,  0.0005, -0.0072],
        ...,
        [ 0.0083, -0.0066, -0.0033,  ..., -0.0047,  0.0012, -0.0070],
        [-0.0161,  0.0008, -0.0009,  ...,  0.0006, -0.0064, -0.0030],
        [ 0.0129,  0.0076,  0.0081,  ...,  0.0055, -0.0123, -0.0147]],
       grad_fn=<PermuteBackward0>)

In [9]:
enc_1

tensor([[ 2.5484e-04,  3.7491e-04, -1.5784e-04,  ..., -6.2764e-04,
         -7.3574e-04, -1.6418e-04],
        [-6.9016e-04, -6.0766e-04, -1.1011e-03,  ...,  3.3418e-05,
         -5.6823e-04, -4.1784e-04],
        [-6.2946e-04, -6.4760e-04,  2.5053e-04,  ..., -5.2962e-04,
          3.1797e-05, -4.5294e-04],
        ...,
        [ 5.2042e-04, -4.1354e-04, -2.0803e-04,  ..., -2.9323e-04,
          7.7461e-05, -4.3737e-04],
        [-1.0093e-03,  5.1901e-05, -5.8346e-05,  ...,  3.9619e-05,
         -4.0438e-04, -1.9153e-04],
        [ 8.1091e-04,  4.8031e-04,  5.0850e-04,  ...,  3.4373e-04,
         -7.7303e-04, -9.2501e-04]], grad_fn=<SelectBackward0>)

In [11]:
inited_cc = ModelHookpointAcausalCrosscoder.load('/home/tim/crosscoder_training/.checkpoints/less_layers_w_n_dead_latents_2025-04-09_21-29-32/epoch_0_step_0', 'cuda')

TypeError: empty() received an invalid combination of arguments - got (tuple, dtype=str), but expected one of:
 * (tuple of ints size, *, tuple of names names, torch.memory_format memory_format = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (tuple of ints size, *, torch.memory_format memory_format = None, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)


In [1]:
import torch

In [2]:
a = torch.tensor([0], dtype=torch.float32)
a.dtype

torch.float32

In [3]:
str(a.dtype)

'torch.float32'

In [4]:
from transformer_lens import HookedTransformer

In [5]:
model = HookedTransformer.from_pretrained('gemma-2-2b')

Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.01s/it]


Loaded pretrained model gemma-2-2b into HookedTransformer


In [6]:
model.cfg

HookedTransformerConfig:
{'NTK_by_parts_factor': 8.0,
 'NTK_by_parts_high_freq_factor': 4.0,
 'NTK_by_parts_low_freq_factor': 1.0,
 'act_fn': 'gelu_pytorch_tanh',
 'attention_dir': 'causal',
 'attn_only': False,
 'attn_scale': np.float64(16.0),
 'attn_scores_soft_cap': 50.0,
 'attn_types': ['global',
                'local',
                'global',
                'local',
                'global',
                'local',
                'global',
                'local',
                'global',
                'local',
                'global',
                'local',
                'global',
                'local',
                'global',
                'local',
                'global',
                'local',
                'global',
                'local',
                'global',
                'local',
                'global',
                'local',
                'global',
                'local',
                'global',
                'local',
          

In [7]:
2304*8

18432

In [9]:
2**14

16384

In [1]:
2**15

32768